# 03 Relation Extraction and Graph Construction

This notebook is self-contained. It builds the core symbol-emotion relation by connecting each symbol to the nearest emotion word in the same poem when the token distance is at most 10.

This is the central research method of the project. The graph does not classify whole poems. Instead, it looks for local poetic associations: a symbol is linked to an emotion only when an emotion word appears very nearby in the poem text.


In [1]:
from pathlib import Path
import json
import pandas as pd

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
PROCESSED_DIR = PROJECT_ROOT / 'data' / 'processed'
GRAPHS_DIR = PROJECT_ROOT / 'outputs' / 'graphs'
POEMS_CLEAN_PATH = PROCESSED_DIR / 'poems_clean.csv'
SYMBOLS_PATH = PROCESSED_DIR / 'extracted_symbols.csv'
EMOTIONS_PATH = PROCESSED_DIR / 'extracted_emotions.csv'
RELATIONS_PATH = PROCESSED_DIR / 'symbol_emotion_edges.csv'
GRAPH_NODES_PATH = PROCESSED_DIR / 'graph_nodes.csv'
GRAPH_EDGES_PATH = PROCESSED_DIR / 'graph_edges.csv'
GRAPH_JSON_PATH = GRAPHS_DIR / 'poetry_graph.json'
MAX_DISTANCE = 10
GRAPHS_DIR.mkdir(parents=True, exist_ok=True)

## Functions

The relation means symbol appears near emotion-related language in this poem, not symbol always means this emotion.

The short distance threshold of 10 tokens is intentionally conservative. A larger window may create more edges, but it also risks connecting words that are not actually related in the poem. The notebook also stores evidence snippets so every edge can be inspected.


In [2]:
def token_context(text, start_token, end_token, window=10):
    tokens = text.split()
    start = max(0, min(start_token, end_token) - window)
    end = min(len(tokens), max(start_token, end_token) + window + 1)
    return ' '.join(tokens[start:end])

def closest_emotions_for_symbol(symbol_row, emotion_rows, max_distance=MAX_DISTANCE):
    if emotion_rows.empty:
        return []
    candidates = emotion_rows.copy()
    candidates['token_distance'] = (candidates['start_token'] - symbol_row.start_token).abs()
    candidates['line_distance'] = (candidates['line_number'] - symbol_row.line_number).abs()
    candidates = candidates[candidates['token_distance'] <= max_distance]
    if candidates.empty:
        return []
    candidates = candidates[candidates['token_distance'] == candidates['token_distance'].min()]
    candidates = candidates[candidates['line_distance'] == candidates['line_distance'].min()]
    return candidates.to_dict('records')

def extract_symbol_emotion_edges(symbols_df, emotions_df, poems_df, max_distance=MAX_DISTANCE):
    poem_text = poems_df.set_index('poem_id')['poem_text'].to_dict()
    rows = []
    for poem_id, symbol_group in symbols_df.groupby('poem_id'):
        emotion_group = emotions_df[emotions_df['poem_id'] == poem_id]
        for symbol in symbol_group.itertuples(index=False):
            for emotion in closest_emotions_for_symbol(symbol, emotion_group, max_distance=max_distance):
                distance = abs(int(symbol.start_token) - int(emotion['start_token']))
                line_distance = abs(int(symbol.line_number) - int(emotion['line_number']))
                rows.append({'source_symbol': symbol.symbol, 'target_emotion': emotion['emotion_category'], 'poem_id': poem_id, 'title': symbol.title, 'author': symbol.author, 'symbol_token': int(symbol.start_token), 'emotion_token': int(emotion['start_token']), 'token_distance': distance, 'line_distance': line_distance, 'symbol_line': int(symbol.line_number), 'emotion_line': int(emotion['line_number']), 'context_snippet': token_context(poem_text.get(poem_id, ''), int(symbol.start_token), int(emotion['start_token']))})
    columns = ['source_symbol', 'target_emotion', 'poem_id', 'title', 'author', 'symbol_token', 'emotion_token', 'token_distance', 'line_distance', 'symbol_line', 'emotion_line', 'context_snippet']
    return pd.DataFrame(rows, columns=columns)

def limited_values(series, limit=5):
    values = []
    for value in series.dropna().astype(str):
        if value and value not in values:
            values.append(value)
        if len(values) == limit:
            break
    return ' | '.join(values)

def aggregate_symbol_emotion_edges(relations_df):
    grouped = relations_df.groupby(['source_symbol', 'target_emotion'], as_index=False).agg(frequency=('poem_id', 'size'), poem_count=('poem_id', 'nunique'), avg_distance=('token_distance', 'mean'), min_distance=('token_distance', 'min'), example_titles=('title', limited_values), example_snippets=('context_snippet', limited_values))
    grouped['score'] = grouped['frequency'] / grouped['avg_distance'].clip(lower=1)
    return grouped.sort_values(['frequency', 'poem_count'], ascending=False)

def build_graph_nodes_edges(poems_df, symbols_df, emotions_df, relations_df):
    node_rows = []
    edge_rows = []
    for symbol, frequency in symbols_df['symbol'].value_counts().items():
        node_rows.append({'id': f'symbol:{symbol}', 'label': symbol, 'type': 'symbol', 'frequency': int(frequency), 'author': ''})
    for emotion, frequency in emotions_df['emotion_category'].value_counts().items():
        node_rows.append({'id': f'emotion:{emotion}', 'label': emotion, 'type': 'emotion', 'frequency': int(frequency), 'author': ''})
    for poem in poems_df.itertuples(index=False):
        node_rows.append({'id': f'poem:{poem.poem_id}', 'label': poem.title or poem.poem_id, 'type': 'poem', 'frequency': 1, 'author': poem.author})
    for author, count in poems_df['author'].fillna('').value_counts().items():
        if author:
            node_rows.append({'id': f'author:{author}', 'label': author, 'type': 'author', 'frequency': int(count), 'author': ''})
    for row in aggregate_symbol_emotion_edges(relations_df).itertuples(index=False):
        edge_rows.append({'source': f'symbol:{row.source_symbol}', 'target': f'emotion:{row.target_emotion}', 'type': 'NEAR_EMOTION', 'weight': int(row.frequency), 'poem_id': '', 'evidence': row.example_snippets})
    for row in symbols_df.itertuples(index=False):
        edge_rows.append({'source': f'poem:{row.poem_id}', 'target': f'symbol:{row.symbol}', 'type': 'HAS_SYMBOL', 'weight': 1, 'poem_id': row.poem_id, 'evidence': ''})
    for row in emotions_df.itertuples(index=False):
        edge_rows.append({'source': f'poem:{row.poem_id}', 'target': f'emotion:{row.emotion_category}', 'type': 'HAS_EMOTION_WORD', 'weight': 1, 'poem_id': row.poem_id, 'evidence': row.matched_word})
    for poem in poems_df.itertuples(index=False):
        if poem.author:
            edge_rows.append({'source': f'poem:{poem.poem_id}', 'target': f'author:{poem.author}', 'type': 'WRITTEN_BY', 'weight': 1, 'poem_id': poem.poem_id, 'evidence': ''})
    return pd.DataFrame(node_rows).drop_duplicates('id'), pd.DataFrame(edge_rows).drop_duplicates(['source', 'target', 'type', 'poem_id'])

def export_graph_json(nodes_df, edges_df, path):
    with open(path, 'w', encoding='utf-8') as handle:
        json.dump({'nodes': nodes_df.to_dict('records'), 'edges': edges_df.to_dict('records')}, handle, ensure_ascii=False, indent=2)

## Build Relations and Graph

The first output is symbol_emotion_edges.csv, which contains one row per local symbol-emotion relation with poem title, author, token distance, line distance, and context snippet.

The second output is the graph representation. It contains symbol nodes, emotion nodes, poem nodes, and author nodes. The most important edge type is NEAR_EMOTION, which aggregates repeated symbol-emotion relations across the corpus.


In [3]:
poems_df = pd.read_csv(POEMS_CLEAN_PATH)
symbols_df = pd.read_csv(SYMBOLS_PATH)
emotions_df = pd.read_csv(EMOTIONS_PATH)
relations_df = extract_symbol_emotion_edges(symbols_df, emotions_df, poems_df, max_distance=MAX_DISTANCE)
relations_df.to_csv(RELATIONS_PATH, index=False)
relations_df.head()

,source_symbol,target_emotion,poem_id,title,author,symbol_token,emotion_token,token_distance,line_distance,symbol_line,emotion_line,context_snippet
0,thatch,loneliness,poem_00000,Objects Used to Prop Open a Window,Michelle Menting,56,60,4,0,17,17,"loud—its hinges clack open, clack shut. Stuffe..."
1,window,freedom,poem_00000,Objects Used to Prop Open a Window,Michelle Menting,69,78,9,2,21,23,because this window is split. It's dividing in...
2,hinge,freedom,poem_00000,Objects Used to Prop Open a Window,Michelle Menting,74,78,4,2,21,23,"It's dividing in two. Velvet moss, sagebrush, ..."
3,window,freedom,poem_00000,Objects Used to Prop Open a Window,Michelle Menting,123,120,3,2,37,35,
4,dog,mortality,poem_00000,Objects Used to Prop Open a Window,Michelle Menting,0,1,1,0,1,1,"Dog bone, stapler, cribbage board, garlic pres..."


In [4]:
nodes_df, edges_df = build_graph_nodes_edges(poems_df, symbols_df, emotions_df, relations_df)
nodes_df.to_csv(GRAPH_NODES_PATH, index=False)
edges_df.to_csv(GRAPH_EDGES_PATH, index=False)
export_graph_json(nodes_df, edges_df, GRAPH_JSON_PATH)
aggregate_symbol_emotion_edges(relations_df).head(20)

,source_symbol,target_emotion,frequency,poem_count,avg_distance,min_distance,example_titles,example_snippets,score
22612,god,faith,1925,289,0.170909,0,Someone once said we were put on this earth to...,"is done, until the creature has been cleaned a...",1925.0
32015,love,love,1911,399,0.184197,0,Book Review: The Mountain Lion by Jean Staffor...,forgotten for all the good I do. Love is sunli...,1911.0
50914,soul,transcendence,1142,184,0.387040,0,"from Aurora Leigh, First Book | A Brief Treati...","fresh-sprinkling dreams, And, rounding to the ...",1142.0
30828,light,hope,995,214,0.273367,0,The Properties of Light | Deed | Picking up Yo...,light shoot from my brother’s forehead as we s...,995.0
13875,death,mortality,749,167,0.152203,0,Don’t Bother the Earth Spirit | The Poem of De...,"deaths of all those you love, the most blindin...",749.0
15879,dream,wonder,721,175,0.386963,0,"loose strife [Say, when we woke those icy spri...",at him directly. His image on the surface of t...,721.0
43702,rain,melancholy,540,136,0.303704,0,From where I stand | The bottoms of my shoes |...,has been washed and rinsed by rain. Beyond the...,540.0
48241,shadow,fear,484,121,0.365702,0,"Man In Boat, 1998 | What I've Come to Discuss ...","shadow; when the shadow drifts, the boat drift...",484.0
1348,angel,transcendence,400,100,0.382500,0,Japanese Poems | The Violent Space (or when yo...,one day. The angel in her credenza of extreme ...,400.0
51442,spirit,transcendence,390,100,0.310256,0,Glossary of Selected Terms | I Look for You | ...,"blood.Universe, if not outermost concentric ci...",390.0
